# Phase 3 — GDELT Disruption Integration

Pulls live maritime-relevant disruption events from the GDELT GEO 2.0 API, geo-matches them to ports within a buffer radius, computes a severity score, and penalizes affected graph edges.

**Fallback built in:** every successful pull is cached to disk. If the live call fails (network, rate limit, empty response), the notebook automatically falls back to the last cached file so the rest of the pipeline never breaks.

In [13]:
import requests
import pandas as pd
import numpy as np
import networkx as nx
import pickle
import json
import os
from datetime import datetime
from geopy.distance import great_circle

PROCESSED_DIR = "../data/processed"
CACHE_DIR = "../data/processed/gdelt_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Load the graph and ports from Phase 1/2
with open(f"{PROCESSED_DIR}/maritime_graph.gpickle", "rb") as f:
    G = pickle.load(f)

ports = pd.read_csv(f"{PROCESSED_DIR}/ports_clean.csv")
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph loaded: 1543 nodes, 9462 edges


In [14]:
import json
import os
from datetime import datetime, timedelta

CACHE_DIR = "../data/processed/gdelt_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
CACHE_FILE = f"{CACHE_DIR}/latest_chokepoint_events.json"
CACHE_MAX_AGE_MINUTES = 30  # reuse cached data if younger than this, avoids hammering the rate limit

GDELT_DOC_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

CHOKEPOINTS = {
    "Suez Canal / Red Sea": {"lat": 15.0, "lon": 42.5},
    "Panama Canal": {"lat": 9.1, "lon": -79.7},
    "Strait of Hormuz": {"lat": 26.5, "lon": 56.3},
    "Strait of Malacca": {"lat": 2.5, "lon": 101.5},
    "Strait of Gibraltar": {"lat": 35.9, "lon": -5.6},
}

# Fallback scenarios (restored — these were dropped when the cell was rewritten)
FALLBACK_SCENARIOS = {
    "red_sea_crisis": [
        {"latitude": 30.5, "longitude": 32.3, "name": "Suez Canal disruption (fallback)", "count": 10, "tone": -8.0},
        {"latitude": 13.0, "longitude": 43.0, "name": "Red Sea disruption (fallback)", "count": 10, "tone": -9.0},
    ],
    "panama_drought": [
        {"latitude": 9.1, "longitude": -79.7, "name": "Panama Canal disruption (fallback)", "count": 8, "tone": -5.0},
    ],
    "combined": [
        {"latitude": 30.5, "longitude": 32.3, "name": "Suez Canal disruption (fallback)", "count": 10, "tone": -8.0},
        {"latitude": 13.0, "longitude": 43.0, "name": "Red Sea disruption (fallback)", "count": 10, "tone": -9.0},
        {"latitude": 9.1, "longitude": -79.7, "name": "Panama Canal disruption (fallback)", "count": 8, "tone": -5.0},
    ]
}
ACTIVE_FALLBACK_SCENARIO = "combined"


def fetch_combined_doc_signal(timespan="1d", max_records=100):
    combined_query = (
        '("suez canal" OR "red sea" OR "panama canal" OR '
        '"strait of hormuz" OR "strait of malacca") sourcelang:english'
    )
    params = {
        "query": combined_query,
        "mode": "artlist",
        "maxrecords": max_records,
        "format": "json",
        "timespan": timespan,
    }
    response = requests.get(GDELT_DOC_URL, params=params, timeout=40)
    response.raise_for_status()
    return response.json().get("articles", [])


def attribute_articles_to_chokepoints(articles):
    chokepoint_keywords = {
        "Suez Canal / Red Sea": ["suez", "red sea", "bab el-mandeb", "houthi"],
        "Panama Canal": ["panama canal"],
        "Strait of Hormuz": ["hormuz"],
        "Strait of Malacca": ["malacca"],
        "Strait of Gibraltar": ["gibraltar"],
    }
    events = []
    for name, info in CHOKEPOINTS.items():
        matched = [a for a in articles if any(kw in a.get("title", "").lower() for kw in chokepoint_keywords[name])]
        if matched:
            tones = [a.get("tone", 0.0) for a in matched if "tone" in a]
            avg_tone = sum(tones) / len(tones) if tones else 0.0
            events.append({
                "latitude": info["lat"], "longitude": info["lon"],
                "name": name, "count": len(matched), "tone": avg_tone,
            })
    return events


def load_cache_if_fresh():
    if not os.path.exists(CACHE_FILE):
        return None
    with open(CACHE_FILE, "r") as f:
        cached = json.load(f)
    cached_time = datetime.fromisoformat(cached["timestamp"])
    if datetime.utcnow() - cached_time < timedelta(minutes=CACHE_MAX_AGE_MINUTES):
        return cached["events"]
    return None


def save_cache(events):
    with open(CACHE_FILE, "w") as f:
        json.dump({"timestamp": datetime.utcnow().isoformat(), "events": events}, f)


def get_chokepoint_events_with_fallback():
    # 1. Try fresh cache first — avoids unnecessary live calls entirely
    cached_events = load_cache_if_fresh()
    if cached_events:
        print(f"Using cached GDELT data (< {CACHE_MAX_AGE_MINUTES} min old): {len(cached_events)} chokepoints")
        return cached_events, "cached"

    # 2. Try live pull
    try:
        articles = fetch_combined_doc_signal()
        if not articles:
            raise ValueError("GDELT returned zero articles")
        events = attribute_articles_to_chokepoints(articles)
        if not events:
            raise ValueError("No articles matched any known chokepoint keywords")
        save_cache(events)
        print(f"Live GDELT DOC pull succeeded: {len(articles)} articles, {len(events)} chokepoints with activity")
        return events, "live"
    except Exception as e:
        print(f"Live GDELT DOC pull failed ({e}).")
        # 3. Fall back to stale cache if it exists, even if older than freshness window
        if os.path.exists(CACHE_FILE):
            with open(CACHE_FILE, "r") as f:
                stale = json.load(f)
            print(f"Using stale cache from {stale['timestamp']} instead.")
            return stale["events"], "stale_cache"
        # 4. Last resort: hardcoded scenario
        print("No cache available. Falling back to hardcoded scenario.")
        return FALLBACK_SCENARIOS[ACTIVE_FALLBACK_SCENARIO], "hardcoded_fallback"


events, source = get_chokepoint_events_with_fallback()
print(f"Source: {source} | Events: {len(events)}")

Live GDELT DOC pull succeeded: 100 articles, 2 chokepoints with activity
Source: live | Events: 2


C:\Users\varch\AppData\Local\Temp\ipykernel_27496\3753430805.py:89: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  json.dump({"timestamp": datetime.utcnow().isoformat(), "events": events}, f)


In [15]:
events_df = pd.DataFrame(events)
print(events_df.shape)
events_df.head(10)

(2, 5)


,latitude,longitude,name,count,tone
0,15.0,42.5,Suez Canal / Red Sea,9,0.0
1,26.5,56.3,Strait of Hormuz,37,0.0


## Geo-match events to ports
For each port, find events within a buffer radius (nautical miles). If any events fall within the buffer, compute a severity score for that port from event count + tone.

In [16]:
BUFFER_NM = 250  # matches the paper's ~200km buffer, converted to nm and rounded

def compute_port_severity(port_lat, port_lon, events_df, buffer_nm=BUFFER_NM):
    """
    Returns a severity score in [0, 1] for a port based on nearby GDELT events.
    Severity combines event count (more events = more severe) and tone
    (more negative tone = more severe), normalized to [0,1].
    """
    if events_df.empty:
        return 0.0

    port_coord = (port_lat, port_lon)
    nearby = []
    for _, ev in events_df.iterrows():
        try:
            dist = great_circle(port_coord, (ev['latitude'], ev['longitude'])).nautical
        except Exception:
            continue
        if dist <= buffer_nm:
            nearby.append(ev)

    if not nearby:
        return 0.0

    nearby_df = pd.DataFrame(nearby)
    total_count = nearby_df['count'].sum()
    avg_tone = nearby_df['tone'].mean()  # typically negative for bad news, e.g. -10 to -5

    # Normalize: count contributes via log-scale (diminishing returns), tone via clipped negative range
    count_component = min(np.log1p(total_count) / np.log1p(50), 1.0)  # saturate around 50 mentions
    tone_component = min(max(-avg_tone / 10.0, 0.0), 1.0)  # -10 tone -> 1.0, 0 tone -> 0.0

    severity = 0.5 * count_component + 0.5 * tone_component
    return round(float(severity), 3)

In [17]:
# Compute severity for every port in the graph
port_severity = {}
for node_id, attrs in G.nodes(data=True):
    sev = compute_port_severity(attrs['latitude'], attrs['longitude'], events_df)
    port_severity[node_id] = sev
    if sev > 0:
        nx.set_node_attributes(G, {node_id: sev}, name='disruption_severity')

# Ports with zero severity still get the attribute, defaulted to 0.0
nx.set_node_attributes(G, {n: port_severity.get(n, 0.0) for n in G.nodes()}, name='disruption_severity')

affected_ports = {k: v for k, v in port_severity.items() if v > 0}
print(f"Ports affected by disruption: {len(affected_ports)} / {G.number_of_nodes()}")
print("Top 10 most severe:")
for port_id, sev in sorted(affected_ports.items(), key=lambda x: -x[1])[:10]:
    print(f"  {G.nodes[port_id]['port_name']} ({G.nodes[port_id]['country']}): {sev}")

Ports affected by disruption: 29 / 1543
Top 10 most severe:
  Sharjah Offshore Terminal (United Arab Emirates): 0.463
  Mina Jabal Ali (United Arab Emirates): 0.463
  Dubayy (United Arab Emirates): 0.463
  Abu Zaby (United Arab Emirates): 0.463
  Jazireh-Ye Sirri (Iran): 0.463
  Jabal Az Zannah/ruways (United Arab Emirates): 0.463
  Al Hamriyah Lpg Terminal (United Arab Emirates): 0.463
  Ash Shariqah (United Arab Emirates): 0.463
  Khawr Fakkan (United Arab Emirates): 0.463
  Mina Al Fahl (Oman): 0.463


## Penalize edges touching disrupted ports
An edge's `disrupted_weight` = `base_weight * (1 + penalty_multiplier * max(severity of the two endpoint ports))`.
This keeps `base_weight` (pure distance) untouched — Baseline 1 can use either depending on whether disruption should apply.

In [18]:
PENALTY_MULTIPLIER = 30.0  

for u, v, data in G.edges(data=True):
    sev_u = G.nodes[u].get('disruption_severity', 0.0)
    sev_v = G.nodes[v].get('disruption_severity', 0.0)
    max_sev = max(sev_u, sev_v)

    penalty_factor = 1.0 + PENALTY_MULTIPLIER * max_sev
    data['disrupted_weight'] = data['base_weight'] * penalty_factor
    data['disruption_severity_edge'] = max_sev

penalized_edges = [(u, v) for u, v, d in G.edges(data=True) if d['disruption_severity_edge'] > 0]
print(f"Edges penalized: {len(penalized_edges)} / {G.number_of_edges()}")

Edges penalized: 243 / 9462


In [19]:
# --- Save the disruption-tagged graph (separate file — keeps Phase 2's clean graph untouched) ---

output_path = f"{PROCESSED_DIR}/maritime_graph_disrupted.gpickle"
with open(output_path, "wb") as f:
    pickle.dump(G, f)

print(f"Disruption-tagged graph saved to {output_path}")
print(f"Data source used: {source}")
print(f"Affected ports: {len(affected_ports)}, Penalized edges: {len(penalized_edges)}")

Disruption-tagged graph saved to ../data/processed/maritime_graph_disrupted.gpickle
Data source used: live
Affected ports: 29, Penalized edges: 243
